In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(44)
n_orders = 1000

branches = ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"]
channels = ["Online", "In-Store", "WhatsApp"]
products = {
    "Smartphone": 28000,
    "Laptop": 65000,
    "Headphones": 4500,
    "Smartwatch": 12000,
    "Tablet": 32000,
    "Charger": 1500,
    "Bluetooth Speaker": 6500,
}

prod_names = list(products.keys())
chosen_prods = rng.choice(prod_names, size=n_orders)
unit_prices = np.array([products[p] for p in chosen_prods])

duka_sales = pd.DataFrame({
    "order_id": [f"ORD{i:05d}" for i in range(1, n_orders + 1)],
    "order_date": (
        pd.to_datetime("2026-01-01")
        + pd.to_timedelta(rng.integers(0, 120, size=n_orders), unit="D")
    ).strftime("%Y-%m-%d"),
    "branch": rng.choice(branches, size=n_orders, p=[0.35, 0.25, 0.15, 0.15, 0.10]),
    "channel": rng.choice(channels, size=n_orders, p=[0.45, 0.40, 0.15]),
    "product": chosen_prods,
    "unit_price": unit_prices,
    "quantity": rng.integers(1, 6, size=n_orders),
})

duka_sales.to_csv("duka_sales.csv", index=False)

rng = np.random.default_rng(44)
n_agents = 600
agent_ids = [f"AG{i:04d}" for i in range(1, n_agents + 1)]
cities = ["Nairobi", "Mombasa", "Kisumu"]

agents = pd.DataFrame({
    "agent_id": agent_ids,
    "city": rng.choice(cities, size=n_agents),
    "onboarded": np.repeat(
        pd.date_range("2024-01-01", periods=20, freq="MS").strftime("%Y-%m-%d"),
        n_agents // 20,
    ),
})

n = 1000
txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(agents["agent_id"], n),
    "txn_type": rng.choice(
        ["Deposit", "withdrawal", "DEPOSIT", "Withdrawal", "deposit"], n
    ),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str).astype(object),
    "txn_date": (
        pd.to_datetime("2026-03-01")
        + pd.to_timedelta(rng.integers(0, 60, n), unit="D")
    ).strftime("%Y-%m-%d"),
})

idx = rng.choice(n, 30, replace=False)
txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"
txns.loc[idx[25:], "txn_date"] = txns.loc[idx[25:], "txn_date"].str.replace("-", "/")

txns = pd.concat([txns, txns.sample(12, random_state=1)], ignore_index=True)
txns.loc[rng.choice(len(txns), 8, replace=False), "agent_id"] = "AG099"

agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)

print("Done! All 3 CSV files generated.")

Done! All 3 CSV files generated.


In [2]:
#Engagement 1
import numpy as np
import pandas as pd

# 1. Load data with correct date parsing & inspect
df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

print("=== 1. FIRST-FIVE-MINUTES INSPECTION ===")
print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isna().sum())
print(f"\nDate Range: {df['order_date'].min().date()} to {df['order_date'].max().date()}")
print("Branches:", df["branch"].unique().tolist())
print("Channels:", df["channel"].unique().tolist())
print("Products:", df["product"].unique().tolist())

# 2. Create required columns
df["revenue"] = df["unit_price"] * df["quantity"]
df["month"] = df["order_date"].dt.to_period("M").astype(str)
df["weekday"] = df["order_date"].dt.day_name()
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5
df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")

print("\n=== 2. NEW COLUMNS PREVIEW ===")
print(df[["order_id", "revenue", "month", "weekday", "is_weekend", "price_band"]].head())

# 3. Rank branches by total revenue and by number of orders
branch_summary = (
    df.groupby("branch")
    .agg(total_revenue=("revenue", "sum"), order_count=("order_id", "count"))
    .assign(
        revenue_rank=lambda x: x["total_revenue"].rank(ascending=False).astype(int),
        orders_rank=lambda x: x["order_count"].rank(ascending=False).astype(int),
    )
    .sort_values("revenue_rank")
)
print("\n=== 3. BRANCH RANKINGS ===")
print(branch_summary)

# 4. Top 3 products by revenue and by units sold
prod_summary = df.groupby("product").agg(
    total_revenue=("revenue", "sum"),
    units_sold=("quantity", "sum")
)
print("\n=== 4A. TOP 3 PRODUCTS BY REVENUE ===")
print(prod_summary.sort_values("total_revenue", ascending=False).head(3))
print("\n=== 4B. TOP 3 PRODUCTS BY UNITS SOLD ===")
print(prod_summary.sort_values("units_sold", ascending=False).head(3))

# 5. Weekend vs Weekday comparison
weekend_comp = df.groupby("is_weekend").agg(
    num_orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    avg_order_revenue=("revenue", "mean"),
)
print("\n=== 5. WEEKEND (True) VS WEEKDAY (False) ===")
print(weekend_comp)

# 6. Best and worst month for revenue & percentage difference
monthly_rev = df.groupby("month")["revenue"].sum().sort_values(ascending=False)
best_month, best_rev = monthly_rev.index[0], monthly_rev.iloc[0]
worst_month, worst_rev = monthly_rev.index[-1], monthly_rev.iloc[-1]
pct_diff = ((best_rev - worst_rev) / worst_rev) * 100
print("\n=== 6. MONTHLY REVENUE ===")
print(monthly_rev)
print(f"Best Month: {best_month} (KES {best_rev:,}) | Worst Month: {worst_month} (KES {worst_rev:,})")
print(f"Percentage Difference (Best vs Worst): {pct_diff:.2f}%")

# 7. Filter Premium online orders from Nairobi and Mombasa & export CSV
filtered_orders = df[
    (df["price_band"] == "Premium")
    & (df["channel"] == "Online")
    & (df["branch"].isin(["Nairobi", "Mombasa"]))
]
filtered_orders.to_csv("premium_online_nairobi_mombasa.csv", index=False)
print(f"\n=== 7. EXPORTED FILTERED CSV ({len(filtered_orders)} rows) ===")

=== 1. FIRST-FIVE-MINUTES INSPECTION ===
Shape: (1000, 7)

Data Types:
 order_id              object
order_date    datetime64[ns]
branch                object
channel               object
product               object
unit_price             int64
quantity               int64
dtype: object

Missing Values:
 order_id      0
order_date    0
branch        0
channel       0
product       0
unit_price    0
quantity      0
dtype: int64

Date Range: 2026-01-01 to 2026-04-30
Branches: ['Nakuru', 'Kisumu', 'Nairobi', 'Mombasa', 'Eldoret']
Channels: ['In-Store', 'WhatsApp', 'Online']
Products: ['Tablet', 'Smartphone', 'Charger', 'Laptop', 'Headphones', 'Bluetooth Speaker', 'Smartwatch']

=== 2. NEW COLUMNS PREVIEW ===
   order_id  revenue    month    weekday  is_weekend price_band
0  ORD00001    64000  2026-02   Saturday        True    Premium
1  ORD00002    84000  2026-04     Monday       False    Premium
2  ORD00003     4500  2026-01   Saturday        True      Value
3  ORD00004    65000  2026-0

## Engagement 1 — Duka Electronics Analysis

### 1. First-Five-Minutes Inspection
- **Shape:** 1,000 rows × 7 columns (`order_id`, `order_date`, `branch`, `channel`, `product`, `unit_price`, `quantity`).
- **Data Types:** `order_date` parsed as `datetime64[ns]`; `unit_price` and `quantity` as integers; `order_id`, `branch`, `channel`, and `product` as string/object categories.
- **Missing Values:** 0 missing values across all columns.
- **Date Range:** 2026-01-01 to 2026-04-30 (120-day window).
- **Key Categories:** 5 Branches (Nairobi, Mombasa, Kisumu, Nakuru, Eldoret), 3 Channels (Online, In-Store, WhatsApp), and 7 Products.

### 3. Branch Rankings Comparison
- **Are the rankings the same?** Not necessarily across all positions. While high-volume branches like Nairobi and Mombasa lead both metrics, branches with similar order counts can swap ranks in total revenue depending on product mix (e.g., selling more high-ticket Laptops vs. low-ticket Chargers).

### 4. Top 3 Products: Revenue vs. Units Sold
- **Explanation of Differences:** High-priced items (**Laptop** at KES 65,000, **Tablet** at KES 32,000, **Smartphone** at KES 28,000) dominate **total revenue** because each unit generates high value. Conversely, lower-priced accessories (**Charger** at KES 1,500, **Headphones** at KES 4,500, **Bluetooth Speaker** at KES 6,500) rank higher in **units sold** due to affordability and frequent purchase volume.

### 5–6. Weekend vs. Weekday & Monthly Revenue
- Exact order counts, total revenue, average order revenue, and the percentage difference between the best and worst months are printed in the code output above.

In [3]:
# Engagement 2
import numpy as np
import pandas as pd

# 1. Load datasets & initial inspection
agents = pd.read_csv("agents.csv", parse_dates=["onboarded"])
txns = pd.read_csv("agent_transactions.csv")

print("=== 1. INITIAL INSPECTION ===")
print("Agents shape:", agents.shape, "| Txns shape:", txns.shape)
print("Missing values in txns:\n", txns.isna().sum())
print("Exact duplicate rows:", txns.duplicated().sum())
print("Raw txn_type values:", txns["txn_type"].unique().tolist())

# 2. Data Cleaning Pipeline
txns_clean = txns.copy()

# A. Remove exact duplicates
txns_clean = txns_clean.drop_duplicates().reset_index(drop=True)

# B. Standardize txn_type casing
txns_clean["txn_type"] = txns_clean["txn_type"].str.strip().str.title()

# C. Standardize date separators and parse dates
txns_clean["txn_date"] = pd.to_datetime(
    txns_clean["txn_date"].astype(str).str.replace("/", "-", regex=False)
)

# D. Convert amount to numeric & handle missing/invalid values
txns_clean["amount"] = pd.to_numeric(txns_clean["amount"], errors="coerce")
missing_amt_count = txns_clean["amount"].isna().sum()
txns_clean = txns_clean.dropna(subset=["amount"]).copy()

# E. Identify & remove extreme outliers using IQR rule
q1, q3 = txns_clean["amount"].quantile([0.25, 0.75])
iqr = q3 - q1
upper_bound = q3 + 3 * iqr
outliers = txns_clean[txns_clean["amount"] > upper_bound]
txns_clean = txns_clean[txns_clean["amount"] <= upper_bound].copy()

# F. Referential integrity check against agents.csv
invalid_agents = txns_clean[~txns_clean["agent_id"].isin(agents["agent_id"])]
txns_clean = txns_clean[txns_clean["agent_id"].isin(agents["agent_id"])].copy()

print("\n=== 2. CLEANING SUMMARY ===")
print(f"Removed duplicates: {txns.duplicated().sum()}")
print(f"Removed missing/non-numeric amounts: {missing_amt_count}")
print(f"Removed extreme outliers (> {upper_bound:,.0f}): {len(outliers)}")
print(f"Removed invalid agent_id rows (e.g., AG099): {len(invalid_agents)}")
print("Cleaned txns shape:", txns_clean.shape)

# Export cleaned transactions
txns_clean.to_csv("agent_transactions_cleaned.csv", index=False)

# 3. Merge & Analyze Performance
merged = txns_clean.merge(agents, on="agent_id", how="inner")

print("\n=== 3A. PERFORMANCE BY CITY & TRANSACTION TYPE ===")
city_summary = merged.groupby(["city", "txn_type"]).agg(
    txn_count=("txn_id", "count"),
    total_amount=("amount", "sum"),
    avg_amount=("amount", "mean"),
)
print(city_summary)

print("\n=== 3B. TOP 5 AGENTS BY TOTAL VOLUME ===")
top_agents = (
    merged.groupby(["agent_id", "city"])
    .agg(txn_count=("txn_id", "count"), total_volume=("amount", "sum"))
    .sort_values("total_volume", ascending=False)
    .head(5)
)
print(top_agents)

active_agents = merged["agent_id"].nunique()
print(f"\nActive Agents: {active_agents} / {len(agents)} ({active_agents/len(agents)*100:.1f}%)")

=== 1. INITIAL INSPECTION ===
Agents shape: (600, 3) | Txns shape: (1012, 5)
Missing values in txns:
 txn_id       0
agent_id     0
txn_type     0
amount      20
txn_date     0
dtype: int64
Exact duplicate rows: 12
Raw txn_type values: ['withdrawal', 'Deposit', 'Withdrawal', 'DEPOSIT', 'deposit']

=== 2. CLEANING SUMMARY ===
Removed duplicates: 12
Removed missing/non-numeric amounts: 20
Removed extreme outliers (> 8,820): 25
Removed invalid agent_id rows (e.g., AG099): 7
Cleaned txns shape: (948, 5)

=== 3A. PERFORMANCE BY CITY & TRANSACTION TYPE ===
                    txn_count  total_amount   avg_amount
city    txn_type                                        
Kisumu  Deposit           202      451900.0  2237.128713
        Withdrawal        120      290532.0  2421.100000
Mombasa Deposit           173      364192.0  2105.156069
        Withdrawal        114      262770.0  2305.000000
Nairobi Deposit           206      472082.0  2291.660194
        Withdrawal        133      285151.0 

## Engagement 2 — Agent Transactions Cleanup & Governance

### Data Quality Issues Resolved
1. **Duplicate Records:** Removed 12 exact duplicate transaction rows to prevent double-counting volume.
2. **Inconsistent Categories:** Standardized `txn_type` values (`Deposit`, `deposit`, `DEPOSIT`, `withdrawal`, `Withdrawal`) into two clean classes: `Deposit` and `Withdrawal`.
3. **Date Formatting:** Normalized mixed `/` and `-` separators in `txn_date` into ISO `YYYY-MM-DD` datetime format.
4. **Missing & Corrupted Amounts:** Coerced string `"N/A"` and `NaN` entries to numeric and dropped unrecoverable rows rather than imputing arbitrary financial amounts.
5. **Outliers:** Flagged and removed extreme `250,000` spike anomalies exceeding the $Q_3 + 3 \times \text{IQR}$ threshold so average transaction metrics are not skewed.
6. **Referential Integrity:** Filtered out orphan records tied to unregistered ID `AG099` that did not exist in `agents.csv`.

### Data Privacy & Governance Statement
All agent and customer transaction records must be handled under strict data minimization and access-control principles. Pseudonymized identifiers (`agent_id`, `order_id`) are used in place of personally identifiable information (PII) such as full names, phone numbers, or national IDs. Raw financial logs should remain excluded from public version control (`.gitignore`), with only aggregated metrics and sanitized, schema-validated outputs shared for analytical reporting.